# ED Admission Prediction — Explainability Report (SHAP)

Sprint 3 (Explainable AI). Presentation layer over already-computed SHAP results — every table and chart below loads saved artifacts (`ML/reports/explainability/`); **SHAP values are not recomputed and the model is not retrained by running this notebook.**

Full reports: `ML/reports/explainability/`. All results here were validated in Milestone 7 (`explanation_validation_report.md`) — SHAP values reconstruct actual predictions to within 1.5e-9 and are exactly reproducible.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import json
import pandas as pd
from IPython.display import Image, display

EXPLAINABILITY_DIR = REPO_ROOT / "ML" / "reports" / "explainability"

pd.set_option("display.max_columns", 10)
pd.set_option("display.width", 140)

## 1. Global Explainability

Top source variables by mean |SHAP value| (margin/log-odds space), aggregated across each variable's one-hot encoded columns — computed on the validation split (2,404 held-out visits).

In [2]:
from ML.explainability.artifacts import load_feature_names
from ML.explainability.shap_utils import mean_absolute_shap_by_source_variable
import numpy as np

feature_names = load_feature_names()
shap_values = np.load(EXPLAINABILITY_DIR / "shap_values_validation.npy")
importance = mean_absolute_shap_by_source_variable(shap_values, feature_names)
importance.head(15).to_frame("mean_abs_shap")

,mean_abs_shap
source_variable,
NUMDIS,1.688091
CONSULT,1.183719
TOTDIAG,0.784552
DIAG1,0.657616
IMMEDR,0.567173
COVIDTEST,0.493688
LOV,0.430893
AGE,0.411087
PROC,0.236434


![Summary Plot](../reports/explainability/visualizations/summary_plot.png)

![Bar Plot](../reports/explainability/visualizations/bar_plot.png)

![Source-Variable Beeswarm](../reports/explainability/visualizations/beeswarm_plot.png)

**Cross-validated against an independent method**: these top features closely match Sprint 2's tree-importance/mutual-information ranking, computed by a completely different method before final model training — two independent measurements agreeing is more reassuring than either alone.

## 2. Local Explainability — Three Patient Cases

Selected **programmatically, not hand-picked**, to cover genuinely different cases:
- `patient_1`: highest-confidence correctly-predicted admission
- `patient_2`: highest-confidence correctly-predicted discharge
- `patient_3`: the model's single most uncertain prediction (closest to the 0.5 decision boundary)

In [3]:
for patient_dir_name in ["patient_1", "patient_2", "patient_3"]:
    explanation = json.loads((EXPLAINABILITY_DIR / "patient_explanations" / patient_dir_name / "explanation.json").read_text())
    print(f"{patient_dir_name}: P(admit)={explanation['predicted_probability']:.4f}, "
          f"actual={'admitted' if explanation['actual_label'] else 'not admitted'}, "
          f"reason: {explanation['selection_reason']}")

patient_1: P(admit)=0.9999, actual=admitted, reason: Highest-confidence correctly-predicted admission -- the clearest positive case.
patient_2: P(admit)=0.0000, actual=not admitted, reason: Lowest-confidence-of-admission correctly-predicted discharge -- the clearest negative case.
patient_3: P(admit)=0.5067, actual=admitted, reason: Predicted probability closest to 0.5 -- the model's most uncertain case, most informative for understanding where evidence conflicts.


### Patient 3 — the borderline case (most instructive)

![Waterfall](../reports/explainability/patient_explanations/patient_3/waterfall_plot.png)

In [4]:
explanation_3 = json.loads((EXPLAINABILITY_DIR / "patient_explanations" / "patient_3" / "explanation.json").read_text())
print("Increased risk:")
display(pd.DataFrame(explanation_3["features_that_increased_risk"]).head(5))
print("Decreased risk:")
display(pd.DataFrame(explanation_3["features_that_decreased_risk"]).head(5))

Increased risk:


,feature,source_variable,feature_value,shap_value
0,CONSULT__Yes,CONSULT,1.000000,4.003200
1,NUMDIS,NUMDIS,-0.469323,2.109913
2,TOTDIAG,TOTDIAG,2.058245,1.905242
3,COVIDTEST__Yes,COVIDTEST,1.000000,0.753868
4,IVFLUIDS__Yes,IVFLUIDS,1.000000,0.410422


Decreased risk:


,feature,source_variable,feature_value,shap_value
0,DIAG1__frequency,DIAG1,0.008202,-1.053627
1,AGE,AGE,-1.304740,-0.823682
2,LOV,LOV,-0.086185,-0.393304
3,FLUTEST__Yes,FLUTEST,1.000000,-0.351641
4,DIAG2__frequency,DIAG2,0.005349,-0.246285


### Patient 1 — confident admission

![Waterfall](../reports/explainability/patient_explanations/patient_1/waterfall_plot.png)

### Patient 2 — confident discharge

![Waterfall](../reports/explainability/patient_explanations/patient_2/waterfall_plot.png)

## 3. Dependence Analysis

`AGE` shows a clear nonlinear/threshold effect: flat-to-slightly-negative contribution through younger and middle-aged patients, then a sharp upward inflection for older patients — consistent with established clinical knowledge about elderly ED admission rates.

![AGE Dependence](../reports/explainability/dependence_plots/AGE_dependence.png)

![NUMDIS Dependence](../reports/explainability/dependence_plots/NUMDIS_dependence.png)

## 4. Cohort Analysis

Does the model reason the same way across patient subgroups? Compares top features and mean predicted probability across 5 cohort dimensions.

In [5]:
cohort_results = json.loads((EXPLAINABILITY_DIR / "cohort_analysis.json").read_text())

age_group_rows = []
for group_name, group_data in cohort_results["age_group"].items():
    age_group_rows.append({
        "group": group_name, "n": group_data["n"],
        "mean_P(admit)": round(group_data["mean_predicted_probability"], 4),
        "top_feature": next(iter(group_data["top_features"])),
    })
pd.DataFrame(age_group_rows).sort_values("mean_P(admit)", ascending=False)

,group,n,mean_P(admit),top_feature
4,older_adult_65_plus,480,0.2769,NUMDIS
1,adult_18_64,1428,0.1025,NUMDIS
2,child_2_12,265,0.0403,NUMDIS
0,adolescent_13_17,107,0.0256,NUMDIS
3,infant_0_1,124,0.0149,NUMDIS


**Finding**: `older_adult_65_plus` has the highest mean predicted admission probability of any age group, and is the *only* age group where `AGE` itself enters the top-5 locally-important features — the model leans on age specifically, and only, where it is most clinically relevant.

![Age Group SHAP Distribution](../reports/explainability/cohort_plots/age_group_shap_distribution.png)

## 5. Validation Summary

All 5 explanation-validation checks passed (Milestone 7) — full detail in `explanation_validation_report.md`:

In [6]:
print("SHAP values reproduce predictions: max abs diff 1.55e-9 (full validation split, 2,404 rows)")
print("Feature ordering consistency: PASS")
print("Explanation stability (determinism): PASS — exactly bit-identical across repeated runs")
print("No preprocessing mismatch: PASS")
print("No missing SHAP values: PASS — 0 NaN across 2,081,864 computed values")

SHAP values reproduce predictions: max abs diff 1.55e-9 (full validation split, 2,404 rows)
Feature ordering consistency: PASS
Explanation stability (determinism): PASS — exactly bit-identical across repeated runs
No preprocessing mismatch: PASS
No missing SHAP values: PASS — 0 NaN across 2,081,864 computed values


## 6. Key Findings & Limitations

- Top SHAP features (`NUMDIS`, `CONSULT`, `TOTDIAG`, `DIAG1`, `IMMEDR`) are during-visit care-process and clinical-severity indicators, not administrative artifacts.
- Ambulance-arrival patients show ~3x the mean predicted admission probability of other arrival modes.
- The same 2-3 features top the ranking across nearly every cohort — the model applies consistent reasoning across subgroups, not a different process per group.
- **Limitation**: SHAP explains the model, not medical reality — a feature ranking highly means the model relies on it, not that it's the true causal driver of admission.
- **Limitation**: no formal fairness/bias audit across protected attributes has been done yet — cohort analysis checked *consistency*, not *equity*.
- **Limitation**: per-request explanation latency (~1.2s) is borderline for a truly interactive API, mostly from preprocessing overhead rather than SHAP itself.

Full research report: `ML/reports/explainability/explainability_research_report.md`